In [1]:
%cd ~/IAA
import copy
import numpy as np
import pandas as pd
from src.config import PART1, PART2
from src.glm import BandedTweedieGLM
from src.pipeline import build_line, log
from src.target import adjust_claims, load_claims, match_claims

AGE = {"motor": ("driver_age", "driver_age_band"), "buildings": ("policyholder_age", "policyholder_age_band"),
       "contents": ("policyholder_age", "policyholder_age_band")}
FULL_CRED, N_BOOT = 1082, 30
c2, c1 = load_claims(PART2["data_dir"]), load_claims(PART1["data_dir"])
rng = np.random.default_rng(0)

def nb_rel(tr, feats, glm_kw):
    m = BandedTweedieGLM(**glm_kw).fit(tr[feats], tr.target / tr.weight, tr.weight)
    r = m.relativities()
    r = r[r.feature == "new_business"].set_index("level").relativity
    return r["True"] / r["False"]

rows = []
for line in ["motor", "buildings", "contents"]:
    e = build_line(line, PART2, c2)                     # buildings target already without fire (Part 2 config)
    te = e["tr"].merge(e["p"][["row_id", "policy_id"]], on="row_id")
    fe = e["feats"]
    r_em = nb_rel(te, fe, PART2["glm"][line])
    pols = te.policy_id.unique()
    ti = te.set_index("policy_id")
    boots = [np.log(nb_rel(ti.loc[rng.choice(pols, len(pols))].reset_index(), fe, PART2["glm"][line])) for _ in range(N_BOOT)]
    se = float(np.std(boots, ddof=1))
    log(f"{line}: emerging NB relativity {r_em:.3f}, bootstrap SE (log) {se:.3f}")

    d1 = build_line(line, PART1, c1)
    t1 = d1["tr"].merge(d1["p"][["row_id", AGE[line][0]]] if AGE[line][0] not in d1["tr"] else d1["p"][["row_id"]], on="row_id")
    t1[AGE[line][1]] = pd.cut(t1[AGE[line][0]], [0, 24, 34, 49, 64, 200], labels=["17-24", "25-34", "35-49", "50-64", "65+"]).astype(str)
    if line in PART2.get("peril_split", {}):            # same basis as emerging: remove the separately priced peril
        per = PART2["peril_split"][line]["peril"]
        m1 = adjust_claims(match_claims(c1, d1["p"], line, True, cap=None), PART1["adj"][line])
        t1["target"] = t1.target - t1.row_id.map(m1[m1.peril == per].groupby("row_id").net_adj.sum()).fillna(0)
    f1 = [f for f in fe if f in t1.columns]
    r_dev = nb_rel(t1, f1, PART1["glm"][line])
    log(f"{line}: developed NB relativity {r_dev:.3f} ({len(f1)} common features)")

    m2 = adjust_claims(match_claims(c2, e["p"], line, True, cap=None), PART2["adj"][line])
    if line in PART2.get("peril_split", {}):
        m2 = m2[m2.peril != PART2["peril_split"][line]["peril"]]
    nb_rows = set(e["p"].loc[e["p"].new_business.astype(bool), "row_id"])
    n = int(((m2.net_adj > 0) & m2.row_id.isin(nb_rows)).sum())
    z = min(1.0, np.sqrt(n / FULL_CRED))
    differs = abs(np.log(r_em) - np.log(r_dev)) > 2 * se
    r_use = np.exp(z * np.log(r_em) + (1 - z) * np.log(r_dev)) if differs else r_em
    rows.append({"line": line, "emerging": r_em, "SE (log)": se, "developed": r_dev,
                 "|diff| / SE": abs(np.log(r_em) - np.log(r_dev)) / se, "differs (>2 SE)": differs,
                 "NB claims": n, "Z": z, "relativity to use": r_use})
print(pd.DataFrame(rows).round(3).to_string(index=False))

/Users/woojinji/IAA
15:57:58 motor: emerging NB relativity 1.110, bootstrap SE (log) 0.028
15:58:26 motor: developed NB relativity 1.086 (42 common features)
15:58:26 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
15:58:26 buildings: fire priced separately at 96.04 per exposure (developed)
15:58:47 buildings: emerging NB relativity 1.059, bootstrap SE (log) 0.045
15:59:09 buildings: developed NB relativity 1.037 (43 common features)
15:59:09 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
15:59:38 contents: emerging NB relativity 1.077, bootstrap SE (log) 0.061
16:00:00 contents: developed NB relativity 0.983 (45 common features)
     line  emerging  SE (log)  developed  |diff| / SE  differs (>2 SE)  NB claims     Z  relativity to use
    motor     1.110     0.028      1.086        0.772            False        513 0.689              1.110
buildings     1.059     0.045      1.037        0.466            False         62 0.239           

In [3]:
import sys
print(sys.executable)
%pip install openpyxl -q
!mv ~/Downloads/office_premium.py tools/
!{sys.executable} tools/office_premium.py --submission submissions/p2_v1.csv

/Users/woojinji/IAA/.venv/bin/python

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
wrote docs/office_premium_reference.md, results/office_premium/reference_20.csv and results/office_premium/reference_20.xlsx
   row_id      line channel  exposure  risk_row  risk_annual  divisor  office_annual  office_row_period  current_premium  office_annual / current
M00161473     motor  broker      0.96    796.49       825.90     0.49        1858.97            1798.97          1131.78                     1.64
M00176777     motor  broker      0.99    640.39       645.69     0.49        1491.19            1480.39           443.61                     3.36
M00185311     motor  direct      0.96    592.46       614.33     0.69        1013.53             981.83           872.31                     1.16
M00167496     motor  broker      0.98    452.24       459.78     0.49      

In [4]:
!{sys.executable} -m src.pipeline --market part2 --out submissions/p2_v1_venv.csv --compare submissions/p2_v1.csv

16:59:18 motor: GLM, k=1.0423; Pareto, alpha 2.05 borrowed: 3 claims above 100,000, loading 1.54 -> factor 1.0037; mean rate 482.72; total 38,138,486; completeness {2023: 0.979, 2024: 0.787} (2s)
16:59:18 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
16:59:18 buildings: fire priced separately at 96.04 per exposure (developed)
16:59:19 buildings: GLM, k=1.1585; no loading; + separate peril rate 96.04; mean rate 346.07; total 16,395,752; completeness {2023: 1.0, 2024: 0.95} (2s)
16:59:19 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
16:59:21 contents: GLM, k=1.1826; no loading; mean rate 91.62; total 5,407,202; completeness {2023: 0.998, 2024: 0.965} (2s)
16:59:21 saved submissions/p2_v1_venv.csv
compare p2_v1.csv vs p2_v1_venv.csv: rows 209,063 of 209,063; totals 59,941,439 vs 59,941,439 (+0.000%); correlation 1.000000; rows differing by > 1%: 0.00%


In [5]:
import numpy as np
import pandas as pd
from pathlib import Path
from src.config import PART1, PART2
from src.pipeline import build_line
from src.target import load_claims, load_policies

ROOT = Path.home() / "IAA"
EXP = pd.read_parquet(PART2["data_dir"] / "expenses.parquet").set_index("item")
PROPERTY = ["buildings", "contents"]

def needed(df, mkt):
    """df: line, channel, exposure, risk (expected claim cost for the row), prem (premium earned by the row)."""
    ex = EXP[mkt]
    comm = df.channel.map(lambda c: ex.get(f"commission_{c}", np.nan))
    cat = np.where(df.line.isin(PROPERTY), ex["cat_reinsurance"], 0.0) * df.risk
    fixed = ex["fixed_expense_per_policy"] * df.exposure
    div = 1 - ex["variable_expense"] - comm - ex["premium_tax"] - ex["target_profit_margin"]
    return df.assign(cat=cat, fixed=fixed, divisor=div, need=(df.risk + cat + fixed) / div)

def summary(df, label):
    ok = df.need.notna()
    out = []
    for keys, g in list(df[ok].groupby("line")) + [("ALL", df[ok])]:
        out.append({"source": label, "line": keys, "premium": g.prem.sum(), "loss ratio": g.risk.sum() / g.prem.sum(),
                    "needed / current": g.need.sum() / g.prem.sum(), "indicated change": g.need.sum() / g.prem.sum() - 1})
    excl = 1 - df.loc[ok, "prem"].sum() / df.prem.sum()
    if excl > 0:
        print(f"{label}: {excl:.1%} of premium excluded (comparison_site commission blank)")
    return pd.DataFrame(out)

def test_book(cfg, sub_path, mkt):
    sub = pd.read_csv(sub_path, dtype={"row_id": str})
    parts = []
    for line in cfg["lines"]:
        t = load_policies(line, "test", cfg["data_dir"])
        parts.append(t[["row_id", "channel", "exposure_clean", "written_premium"]].assign(line=line))
    t = pd.concat(parts).merge(sub, on="row_id")
    df = pd.DataFrame({"line": t.line, "channel": t.channel, "exposure": t.exposure_clean,
                       "risk": t.prediction, "prem": t.written_premium * t.exposure_clean})
    return needed(df, mkt)

def train_book(cfg, mkt):
    claims = load_claims(cfg["data_dir"])
    parts = []
    for line in cfg["lines"]:
        d = build_line(line, cfg, claims)
        tr = d["tr"].merge(d["p"][["row_id", "channel"]], on="row_id")
        actual = tr["target_all"] if "target_all" in tr else tr["target"]
        parts.append(pd.DataFrame({"line": line, "channel": tr.channel, "exposure": tr.exposure_clean,
                                   "risk": actual * tr.exposure_clean / tr.weight,          # gross up for immaturity
                                   "prem": tr.written_premium * tr.exposure_clean}))
    return needed(pd.concat(parts), mkt)

R = pd.concat([
    summary(test_book(PART2, ROOT / "submissions" / "p2_v1.csv", "emerging"), "emerging, model (test 2025)"),
    summary(train_book(PART2, "emerging"), "emerging, actual (2023-24)"),
    summary(test_book(PART1, ROOT / "submissions" / "v12_freqsev.csv", "developed"), "developed, model (test 2025)"),
    summary(train_book(PART1, "developed"), "developed, actual (2021-24)"),
])
pd.set_option("display.width", 200)
print(R.round(3).to_string(index=False))

emerging, model (test 2025): 4.7% of premium excluded (comparison_site commission blank)


AttributeError: 'DataFrame' object has no attribute 'channel'

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path
from src.config import PART1, PART2
from src.pipeline import build_line
from src.target import load_claims, load_policies

ROOT = Path.home() / "IAA"
EXP = pd.read_parquet(PART2["data_dir"] / "expenses.parquet").set_index("item")
PROPERTY = ["buildings", "contents"]

def needed(df, mkt):
    """df: line, channel, exposure, risk (expected claim cost for the row), prem (premium earned by the row)."""
    ex = EXP[mkt]
    comm = df.channel.map(lambda c: ex.get(f"commission_{c}", np.nan))
    cat = np.where(df.line.isin(PROPERTY), ex["cat_reinsurance"], 0.0) * df.risk
    fixed = ex["fixed_expense_per_policy"] * df.exposure
    div = 1 - ex["variable_expense"] - comm - ex["premium_tax"] - ex["target_profit_margin"]
    return df.assign(cat=cat, fixed=fixed, divisor=div, need=(df.risk + cat + fixed) / div)

def summary(df, label):
    ok = df.need.notna()
    out = []
    for keys, g in list(df[ok].groupby("line")) + [("ALL", df[ok])]:
        out.append({"source": label, "line": keys, "premium": g.prem.sum(), "loss ratio": g.risk.sum() / g.prem.sum(),
                    "needed / current": g.need.sum() / g.prem.sum(), "indicated change": g.need.sum() / g.prem.sum() - 1})
    excl = 1 - df.loc[ok, "prem"].sum() / df.prem.sum()
    if excl > 0:
        print(f"{label}: {excl:.1%} of premium excluded (comparison_site commission blank)")
    return pd.DataFrame(out)

def test_book(cfg, sub_path, mkt):
    sub = pd.read_csv(sub_path, dtype={"row_id": str})
    parts = []
    for line in cfg["lines"]:
        t = load_policies(line, "test", cfg["data_dir"])
        parts.append(t[["row_id", "channel", "exposure_clean", "written_premium"]].assign(line=line))
    t = pd.concat(parts).merge(sub, on="row_id")
    df = pd.DataFrame({"line": t.line, "channel": t.channel, "exposure": t.exposure_clean,
                       "risk": t.prediction, "prem": t.written_premium * t.exposure_clean})
    return needed(df, mkt)

def train_book(cfg, mkt):
    claims = load_claims(cfg["data_dir"])
    parts = []
    for line in cfg["lines"]:
        d = build_line(line, cfg, claims)
        tr = d["tr"].merge(d["p"][["row_id", "channel"]].rename(columns={"channel": "ch"}), on="row_id")
        actual = tr["target_all"] if "target_all" in tr else tr["target"]
        parts.append(pd.DataFrame({"line": line, "channel": tr.ch, "exposure": tr.exposure_clean,
                                   "risk": actual * tr.exposure_clean / tr.weight,          # gross up for immaturity
                                   "prem": tr.written_premium * tr.exposure_clean}))
    return needed(pd.concat(parts), mkt)

R = pd.concat([
    summary(test_book(PART2, ROOT / "submissions" / "p2_v1.csv", "emerging"), "emerging, model (test 2025)"),
    summary(train_book(PART2, "emerging"), "emerging, actual (2023-24)"),
    summary(test_book(PART1, ROOT / "submissions" / "v12_freqsev.csv", "developed"), "developed, model (test 2025)"),
    summary(train_book(PART1, "developed"), "developed, actual (2021-24)"),
])
pd.set_option("display.width", 200)
print(R.round(3).to_string(index=False))

emerging, model (test 2025): 4.7% of premium excluded (comparison_site commission blank)
17:05:44 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
17:05:44 buildings: fire priced separately at 96.04 per exposure (developed)
17:05:45 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
emerging, actual (2023-24): 4.9% of premium excluded (comparison_site commission blank)
                      source      line      premium  loss ratio  needed / current  indicated change
 emerging, model (test 2025) buildings 1.747086e+07       0.891             2.296             1.296
 emerging, model (test 2025)  contents 7.700135e+06       0.668             2.580             1.580
 emerging, model (test 2025)     motor 5.112816e+07       0.707             1.570             0.570
 emerging, model (test 2025)       ALL 7.629915e+07       0.745             1.838             0.838
  emerging, actual (2023-24) buildings 7.965313e+06       1.165             2.925  

In [7]:
import numpy as np
import pandas as pd
from pathlib import Path
from src.config import PART1, PART2
from src.pipeline import build_line
from src.target import load_claims, load_policies

ROOT = Path.home() / "IAA"
EXP = pd.read_parquet(PART2["data_dir"] / "expenses.parquet").set_index("item")
PROPERTY = ["buildings", "contents"]

def needed(df, mkt):
    """df: line, channel, exposure, risk (expected claim cost for the row), prem (premium earned by the row)."""
    ex = EXP[mkt]
    comm = df.channel.map(lambda c: ex.get(f"commission_{c}", np.nan))
    cat = np.where(df.line.isin(PROPERTY), ex["cat_reinsurance"], 0.0) * df.risk
    fixed = ex["fixed_expense_per_policy"] * df.exposure
    div = 1 - ex["variable_expense"] - comm - ex["premium_tax"] - ex["target_profit_margin"]
    return df.assign(cat=cat, fixed=fixed, divisor=div, need=(df.risk + cat + fixed) / div)

def summary(df, label):
    ok = df.need.notna()
    out = []
    for keys, g in list(df[ok].groupby("line")) + [("ALL", df[ok])]:
        out.append({"source": label, "line": keys, "premium": g.prem.sum(), "loss ratio": g.risk.sum() / g.prem.sum(),
                    "needed / current": g.need.sum() / g.prem.sum(), "indicated change": g.need.sum() / g.prem.sum() - 1})
    excl = 1 - df.loc[ok, "prem"].sum() / df.prem.sum()
    if excl > 0:
        print(f"{label}: {excl:.1%} of premium excluded (comparison_site commission blank)")
    return pd.DataFrame(out)

def test_book(cfg, sub_path, mkt):
    sub = pd.read_csv(sub_path, dtype={"row_id": str})
    parts = []
    for line in cfg["lines"]:
        t = load_policies(line, "test", cfg["data_dir"])
        parts.append(t[["row_id", "channel", "exposure_clean", "written_premium"]].assign(line=line))
    t = pd.concat(parts).merge(sub, on="row_id")
    df = pd.DataFrame({"line": t.line, "channel": t.channel, "exposure": t.exposure_clean,
                       "risk": t.prediction, "prem": t.written_premium * t.exposure_clean})
    return needed(df, mkt)

def train_book(cfg, mkt):
    claims = load_claims(cfg["data_dir"])
    parts = []
    for line in cfg["lines"]:
        d = build_line(line, cfg, claims)
        tr = d["tr"].merge(d["p"][["row_id", "channel"]].rename(columns={"channel": "ch"}), on="row_id")
        actual = tr["target_all"] if "target_all" in tr else tr["target"]
        parts.append(pd.DataFrame({"line": line, "channel": tr.ch, "exposure": tr.exposure_clean,
                                   "risk": actual * tr.exposure_clean / tr.weight,          # gross up for immaturity
                                   "prem": tr.written_premium * tr.exposure_clean}))
    return needed(pd.concat(parts), mkt)

R = pd.concat([
    summary(test_book(PART2, ROOT / "submissions" / "p2_v1.csv", "emerging"), "emerging, model (test 2025)"),
    summary(train_book(PART2, "emerging"), "emerging, actual (2023-24)"),
    summary(test_book(PART1, ROOT / "submissions" / "v12_freqsev.csv", "developed"), "developed, model (test 2025)"),
    summary(train_book(PART1, "developed"), "developed, actual (2021-24)"),
])
pd.set_option("display.width", 200)
print(R.round(3).to_string(index=False))

emerging, model (test 2025): 4.7% of premium excluded (comparison_site commission blank)
17:06:46 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
17:06:46 buildings: fire priced separately at 96.04 per exposure (developed)
17:06:46 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
emerging, actual (2023-24): 4.9% of premium excluded (comparison_site commission blank)
                      source      line      premium  loss ratio  needed / current  indicated change
 emerging, model (test 2025) buildings 1.747086e+07       0.891             2.296             1.296
 emerging, model (test 2025)  contents 7.700135e+06       0.668             2.580             1.580
 emerging, model (test 2025)     motor 5.112816e+07       0.707             1.570             0.570
 emerging, model (test 2025)       ALL 7.629915e+07       0.745             1.838             0.838
  emerging, actual (2023-24) buildings 7.965313e+06       1.165             2.925  

In [8]:
import pandas as pd
from src.config import PART2
from src.glm import BandedTweedieGLM
from src.pipeline import build_line
from src.target import adjust_claims, load_claims, match_claims

c2 = load_claims(PART2["data_dir"])
e = build_line("buildings", PART2, c2)                      # target = all perils except fire
tr, feats = e["tr"], e["feats"]
t, v = tr[tr.year == 2023], tr[tr.year == 2024]
m2 = adjust_claims(match_claims(c2, e["p"], "buildings", True, cap=None), PART2["adj"]["buildings"])
per = m2.pivot_table(index="row_id", columns="peril", values="net_adj", aggfunc="sum").fillna(0)
perils = list(per.columns)

for alpha in [2, 10]:
    g = BandedTweedieGLM(**dict(PART2["glm"]["buildings"], alpha=alpha)).fit(t[feats], t.target / t.weight, t.weight)
    vv = v.assign(pred=g.predict_rate(v[feats]) * v.weight.to_numpy())
    vv = vv.merge(per, left_on="row_id", right_index=True, how="left").fillna({p_: 0 for p_ in perils})
    vv["decile"] = pd.qcut((vv.pred / vv.weight).rank(method="first"), 10, labels=range(1, 11))
    agg = vv.groupby("decile", observed=True)[["weight", "pred", "target"] + perils].sum()
    out = pd.DataFrame({"pred rate": agg.pred / agg.weight, "actual non-fire": agg.target / agg.weight,
                        "A/E non-fire": agg.target / agg.pred})
    for p_ in perils:
        out[p_] = agg[p_] / agg.weight
    print(f"\n======== buildings 2024 by predicted decile (fit 2023, alpha {alpha}); peril columns = actual cost per weight")
    print(out.round(2).to_string())

17:13:37 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
17:13:38 buildings: fire priced separately at 96.04 per exposure (developed)

======== buildings 2024 by predicted decile (fit 2023, alpha 2); peril columns = actual cost per weight
        pred rate  actual non-fire  A/E non-fire  accidental_damage  escape_of_water    fire   flood  storm  subsidence
decile                                                                                                                 
1           73.75           111.33          1.51               2.00            65.88   51.38    0.00  43.45        0.00
2          107.43           107.77          1.00               3.50            25.39   43.68    0.00  61.01       17.86
3          133.09           130.83          0.98               7.49            38.19   22.02    0.00  65.33       19.83
4          158.80           135.43          0.85               2.63            58.70  281.69   37.31  36.79        0.00
5          186.0

In [9]:
import copy
import pandas as pd
from src.config import PART1
from src.pipeline import rel_premium, validate
from src.target import load_policies

p = load_policies("contents", "train", PART1["data_dir"])
print("======== Part 1 contents: median written_premium by start month (rows) and year (columns)")
print(p.assign(month=p.period_start.dt.month).pivot_table(index="month", columns="year", values="written_premium",
                                                          aggfunc="median").round(1).to_string())

defs = {n: rel_premium(p, n) for n in ["year", "month", "annual"]}
print("\n======== rank correlation between definitions")
print(pd.DataFrame(defs).corr(method="spearman").round(3).to_string())
print("\n======== mean of each definition by start month (a correct definition is flat)")
print(pd.DataFrame(defs).groupby(p.period_start.dt.month).mean().round(2).T.to_string())

res = []
for norm in ["year", "month", "annual"]:
    cfg = copy.deepcopy(PART1)
    cfg["premium_norm"] = norm
    res.append(validate(cfg, ["contents"]).assign(premium_norm=norm))
print("\n======== Part 1 contents validation by premium definition")
print(pd.concat(res)[["premium_norm", "line", "year", "D2", "A/E", "low", "top"]].round(4).to_string(index=False))

======== Part 1 contents: median written_premium by start month (rows) and year (columns)
year    2021   2022   2023   2024
month                            
1      117.6  124.1  130.9  137.8
2      115.6  122.6  129.9  136.9
3      116.1  124.6  128.1  137.4
4      117.6  122.0  130.0  136.8
5      116.5  122.4  130.6  135.5
6      117.2  122.8  129.9  136.8
7      116.0  123.2  130.2  135.2
8      115.4  120.8  128.6  137.2
9      118.3  122.2  128.7  136.2
10     117.2  122.7  129.6  137.1
11     118.5  122.5  130.4  134.4
12     114.8  122.4  130.6  136.3

======== rank correlation between definitions
         year  month  annual
year    1.000  0.854   0.673
month   0.854  1.000   0.812
annual  0.673  0.812   1.000

======== mean of each definition by start month (a correct definition is flat)
period_start    1     2     3     4     5     6     7     8     9     10    11     12
year          1.82  1.91  1.79  1.77  1.82  2.01  2.24  2.58  3.39  4.69  7.67  33.76
month         2.00 

In [10]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from src.config import PART1, PART2
from src.evaluate import evaluate
from src.glm import BandedTweedieGLM
from src.interactions import add_pairs, pair_name
from src.pipeline import build_line, log
from src.target import adjust_claims, load_claims, match_claims

ALPHAS = [0.5, 2, 10, 50, 200, 1000]
c2 = load_claims(PART2["data_dir"])
e = build_line("buildings", PART2, c2)                     # target = all perils except fire; target_all = all
tr = e["tr"].merge(e["p"][["row_id", "policy_id"]], on="row_id")
m2 = adjust_claims(match_claims(c2, e["p"], "buildings", True, cap=None), PART2["adj"]["buildings"])
tr["flood"] = tr.row_id.map(m2[m2.peril == "flood"].groupby("row_id").net_adj.sum()).fillna(0)
tr["other"] = tr.target - tr.flood
feats, fire_rate = e["feats"], e["sep_rate"]
pairs = [tuple(x) for x in PART1["interactions"]["buildings"]]
t, v = tr[tr.year == 2023].copy(), tr[tr.year == 2024].copy()
t, v = add_pairs(t, [t, v], pairs)                          # bands learned on 2023
PAIRS = [pair_name(a, b) for a, b in pairs]
FLOOD_F = [c for c in ["flood_zone", "elevation_m", "distance_to_river_km", "rainfall_mm", "basement", "property_type",
                       "sum_insured", "building_age", "urban_density", "soil_type", "construction_type",
                       "voluntary_excess"] if c in feats]

def fit_pred(tt, X, f, col, a):
    g = BandedTweedieGLM(**dict(PART2["glm"]["buildings"], alpha=a)).fit(tt[f], tt[col] / tt.weight, tt.weight)
    return g.predict_rate(X[f]) * X.weight.to_numpy()

def cv_alpha(f, col):
    res = {a: np.mean([evaluate(t.iloc[j][col], fit_pred(t.iloc[i], t.iloc[j], f, col, a), t.iloc[j].weight, show=False)["D2"]
                       for i, j in GroupKFold(5).split(t, groups=t.policy_id)]) for a in ALPHAS}
    a = max(res, key=res.get)
    return a

def scores(pv):
    va = v.assign(target=v.target_all)
    r = evaluate(va.target, pv, va.weight, show=False)
    lf, nb = r["lift"]["A/E"], va.new_business.astype(bool).to_numpy()
    return {"D2": r["D2"], "A/E": r["A/E"], "low": lf.iloc[0], "top": lf.iloc[-1],
            "A/E new": va.target.to_numpy()[nb].sum() / pv[nb].sum()}

fire = fire_rate * v.weight.to_numpy()
rows = []
a = cv_alpha(feats, "target");               rows.append({"variant": f"A baseline (alpha {a})", **scores(fit_pred(t, v, feats, "target", a) + fire)})
a = cv_alpha(feats + PAIRS, "target");       rows.append({"variant": f"B + pairs (alpha {a})", **scores(fit_pred(t, v, feats + PAIRS, "target", a) + fire)})
af = cv_alpha(FLOOD_F + PAIRS, "flood");     ao = cv_alpha(feats, "other")
rows.append({"variant": f"C flood GLM+pairs ({af}) + other ({ao})",
             **scores(fit_pred(t, v, FLOOD_F + PAIRS, "flood", af) + fit_pred(t, v, feats, "other", ao) + fire)})
af2 = cv_alpha(FLOOD_F, "flood")
rows.append({"variant": f"D flood GLM ({af2}) + other ({ao})",
             **scores(fit_pred(t, v, FLOOD_F, "flood", af2) + fit_pred(t, v, feats, "other", ao) + fire)})
R = pd.DataFrame(rows)
base = R.iloc[0]
R["passes rule"] = [(r.D2 >= base.D2 + 0.0005) and (abs(r.low - 1) <= abs(base.low - 1) + 0.1)
                    and (abs(r.top - 1) <= abs(base.top - 1) + 0.1) for r in R.itertuples()]
R.loc[0, "passes rule"] = None
print("flood features:", FLOOD_F)
print(R.round(4).to_string(index=False))

17:18:40 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
17:18:41 buildings: fire priced separately at 96.04 per exposure (developed)


TypeError: Invalid value 'nan' for dtype 'bool'

In [11]:
R["passes rule"] = R["passes rule"].astype(object)
R.loc[0, "passes rule"] = "baseline"
print("flood features:", FLOOD_F)
print(R.round(4).to_string(index=False))

flood features: ['flood_zone', 'elevation_m', 'distance_to_river_km', 'rainfall_mm', 'basement', 'property_type', 'sum_insured', 'building_age', 'urban_density', 'soil_type', 'construction_type', 'voluntary_excess']
                             variant     D2    A/E    low    top  A/E new passes rule
               A baseline (alpha 10) 0.0257 0.9095 0.6620 1.6604   0.5956    baseline
                B + pairs (alpha 10) 0.0411 0.9077 0.3117 1.5860   0.5965       False
C flood GLM+pairs (0.5) + other (10) 0.0631 0.9055 0.5135 0.7672   0.6616       False
      D flood GLM (0.5) + other (10) 0.0433 0.9080 0.5338 0.7995   0.6778       False


In [12]:
from pathlib import Path
root = Path.home() / "IAA"

P1_FIX = """### Part 1 correction: relative premium (Oct 23) [notebook 22]
- written_premium is the full annual premium; median written premium in Part 1 is flat by start month every year
  (contents 2021: 115–118) and rises about 5–6% a year. There was no mid-2021 rate step: the policy scan divided
  premium by exposure (December rows ×34). Contents validation D² is the same under all three definitions
  (year 0.0802, month 0.0804, annual 0.0802; mean of 2023 and 2024), so v10's result stands but its stated
  reason was wrong. [AI 100]

"""

E12 = """### E12: buildings flood (Oct 23 evening) — not adopted [notebook 22]
- Diagnosis: the under-priced highest decile (A/E 1.66) is flood: 610–670 of 790–870 actual cost per weight
  in decile 10, almost none in deciles 1–4. One ridge strength cannot fit flood (concentrated) and the other
  perils (alpha 2 fixed decile 10 but broke deciles 7–9). [AI 101]
- Rule (fixed before running): Part 1 GLM-setting rule on 2024 (all perils): D² +0.0005 and lowest/highest
  decile A/E no more than 0.1 further from 1 than the baseline; alphas by CV inside 2023.
- Result: A baseline 0.0257 (low 0.66, top 1.66); B + Part 1 flood pairs 0.0411 (low 0.31); C flood GLM with
  pairs + other-perils GLM 0.0631 (low 0.51, top 0.77); D flood GLM without pairs 0.0433 (low 0.53, top 0.80).
  All fail the lowest-decile condition (C by 0.15). **Not adopted; p2_v1 stays.**
- The gain forgone is large (C: D² +0.037 on buildings); the condition protects the safest customers from
  over-pricing, the team's stated priority. Hypothesis to test as a new, pre-registered experiment: the flat
  fire rate (96 for every policy) drives the low-decile over-prediction in every variant.

"""

AI_ROWS = """| 100 | E | AI's Part 1 policy scan reported a mid-2021 rate rise of about 50% in premium per year, and v10 normalised the relative premium by start month for that reason. | Part 2 showed written_premium is the annual premium; Part 1 medians by year and start month are flat within each year. | No rate step existed (division by exposure inflated late-starting rows). The three definitions give the same contents D² (within 0.0003), so v10's result stands; its stated reason is corrected in DECISIONS. |
| 101 | A | AI split the buildings validation by predicted decile and peril to locate the under-priced highest decile. | Decile × peril table on 2024 (fit 2023), alpha 2 and 10. | The gap is flood (610–670 of 790–870 per weight in decile 10). A flood GLM with Part 1's flood pairs raised D² from 0.026 to 0.063 but over-priced the safest decile (A/E 0.51 vs 0.66) and failed the pre-set rule; not adopted (E12). |
"""

d = root / "DECISIONS.md"
s = d.read_text()
assert "### E12: buildings flood" not in s, "E12 already there"
i = s.find("### Submission p2_v1")
assert i >= 0, "'### Submission p2_v1' not found"
d.write_text(s[:i] + P1_FIX + E12 + s[i:])
print("DECISIONS.md: inserted before ### Submission p2_v1")

a = root / "AI_USE_LOG.md"
s = a.read_text()
assert "| 100 |" not in s, "entry 100 already there"
lines = s.splitlines(keepends=True)
k = max(n for n, l in enumerate(lines) if l.startswith("| 99 |"))
a.write_text("".join(lines[:k + 1]) + AI_ROWS + "".join(lines[k + 1:]))
print("AI_USE_LOG.md: entries 100-101 added after 99")

AssertionError: E12 already there

In [13]:
d = root / "DECISIONS.md"
s = d.read_text()
has_e12, has_fix = "### E12: buildings flood" in s, "### Part 1 correction: relative premium" in s
print("DECISIONS: E12 present:", has_e12, "| Part 1 correction present:", has_fix)
if not has_fix:
    i = s.find("### E12: buildings flood") if has_e12 else s.find("### Submission p2_v1")
    assert i >= 0, "insertion point not found"
    s = s[:i] + P1_FIX + s[i:]
if not has_e12:
    i = s.find("### Submission p2_v1")
    assert i >= 0, "'### Submission p2_v1' not found"
    s = s[:i] + E12 + s[i:]
d.write_text(s)
print("DECISIONS: E12 count", s.count("### E12: buildings flood"), "| correction count", s.count("### Part 1 correction"))

a = root / "AI_USE_LOG.md"
s = a.read_text()
if "| 100 |" in s:
    print("AI_USE_LOG: entry 100 already there")
else:
    lines = s.splitlines(keepends=True)
    k = max(n for n, l in enumerate(lines) if l.startswith("| 99 |"))
    a.write_text("".join(lines[:k + 1]) + AI_ROWS + "".join(lines[k + 1:]))
    print("AI_USE_LOG: entries 100-101 added after 99")

DECISIONS: E12 present: True | Part 1 correction present: False
DECISIONS: E12 count 1 | correction count 1
AI_USE_LOG: entries 100-101 added after 99


In [14]:
import sys
!mv ~/Downloads/exp_blackbox_p2.py tools/
!{sys.executable} tools/exp_blackbox_p2.py --validate

18:15:46 motor ours (p2_v1): D2 0.0177 low 0.749 top 0.976 (1s)
18:15:55 motor black box max: D2 0.0176 low 0.964 top 1.403 (8s)
18:15:55 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
18:15:55 buildings: fire priced separately at 96.04 per exposure (developed)
18:15:56 buildings ours (p2_v1): D2 0.0257 low 0.662 top 1.660 (1s)
18:15:56 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
18:16:01 buildings black box max: D2 -0.0182 low 0.396 top 0.549 (5s)
18:16:01 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
18:16:02 contents ours (p2_v1): D2 0.0316 low 1.036 top 0.920 (1s)
18:16:02 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
18:16:08 contents black box max: D2 0.0055 low 1.223 top 1.137 (6s)

======== fit 2023 -> validate 2024, all perils (low / top = A/E of safest / riskiest 10%)
        line         model      D2    A/E    low    top  A/E new  seed spread
       motor  ours (p2_v1)

In [15]:
import sys
!mv ~/Downloads/exp_trees_p2.py tools/
!{sys.executable} tools/exp_trees_p2.py

18:21:30 motor done (4s)
18:21:30 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
18:21:33 buildings done (2s)
18:21:33 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
18:21:35 contents done (2s)

======== screening (fit 2023, check 2024); D2 after scaling to the 2024 total
     line             peril      part  claims 2023  claims 2024  2024 D2 depth1  2024 D2 depth2  interaction gain  goes to GLM test
    motor     bodily_injury frequency          247          173          0.0258          0.0188           -0.0069             False
    motor     bodily_injury  severity          247          173         -0.0638         -0.1100           -0.0461             False
    motor        glass_hail frequency          375          475          0.0876          0.0833           -0.0043             False
    motor        glass_hail  severity          375          475         -0.0008         -0.0581           -0.0572             False
    motor       

In [16]:
from pathlib import Path
d = Path.home() / "IAA" / "DECISIONS.md"
E9P2 = """### E9-P2: black box max vs ours on the emerging market (Oct 23 evening) — benchmark only [tools/exp_blackbox_p2.py]
- Same as E9 (LightGBM in every line, 127 leaves, 3 seeds, every non-leaking feature, no peril split, no hand-built
  interactions); trees chosen on a 20% policy holdout inside the training year. Fit 2023, validate 2024, all perils.
- D²: motor 0.0177 (ours) vs 0.0176 (black box); buildings 0.0257 vs −0.0182; contents 0.0316 vs 0.0055;
  pooled 0.0806 vs 0.0623. Black box A/E: buildings 0.60 (it learned the 2023 fire-cluster level as normal and
  over-priced 2024 buildings by about two thirds), pooled 0.84.
- Tails (lowest / highest decile A/E): buildings 0.40 / 0.55 (ours 0.66 / 1.66); motor 0.96 / 1.40 (ours
  0.75 / 0.98); contents 1.22 / 1.14 (ours 1.04 / 0.92). Seed spread per row 8.5–10% (ours 0).
- Against the prediction written on Oct 22: "the black box scores higher" did not hold (nor in Part 1, E9);
  "its errors fall on the safest customers" held for buildings (safest 10% priced 2.5× actual) but not
  consistently in motor and contents. The fire-cluster area check (hail days ≤ 16.25, 59% of weight) did not
  separate the models; the evidence is the buildings level.
- Caveat: one training year in this fold; the final models use two.

"""
s = d.read_text()
assert "### E9-P2:" not in s, "E9-P2 already there"
i = s.find("### Submission p2_v1")
assert i >= 0, "'### Submission p2_v1' not found"
d.write_text(s[:i] + E9P2 + s[i:])
print("inserted E9-P2 before ### Submission p2_v1 (after E12)")

inserted E9-P2 before ### Submission p2_v1 (after E12)


In [17]:
from pathlib import Path
d = Path.home() / "IAA" / "DECISIONS.md"
E13 = """### E13: shallow trees on frequency and severity by peril (Oct 23 evening) — screening [tools/exp_trees_p2.py]
- Per line and peril: LightGBM depth 1 (main effects) vs depth 2 (pairs), fit 2023, checked on 2024 (D² after scaling
  to the 2024 total). Frequency on all rows; severity on claims with net cost > 0 only. Screening rule (set before
  running): pairs go to a GLM test if the 2024 gain is ≥ +0.002 (frequency) or ≥ +0.005 (severity).
- Severity: all nine peril models have negative 2024 D², depth 1 included (−0.01 to −0.08): within a peril, claim size
  is not predictable from the rating factors. Severity is effectively a peril average.
- Frequency pairs that hold out of time: motor own_damage +0.0120 (cover_type × voluntary_excess, cover_type ×
  years_licensed, cover_type × ncb_years; own damage is covered only under comprehensive cover) and motor
  third_party_prop +0.0033 (ncb_years × years_licensed, driver_age_band × gender). All other perils negative.
- Flood frequency is strongly predictable from main effects alone (depth-1 D² 0.25 buildings, 0.16 contents: elevation,
  flood zone, distance to river); pairs add nothing (−0.006). Buildings fire (−0.04) and escape of water have no
  predictable structure, supporting the flat fire rate.
- Next (to test under the Part 1 rule): motor own_damage structure (E14); a buildings flood GLM without pairs once the
  team decides how flat-rated parts enter the tail condition (E12); peril frequency × peril-average severity.

"""
s = d.read_text()
assert "### E13:" not in s, "E13 already there"
i = s.find("### Submission p2_v1")
assert i >= 0, "'### Submission p2_v1' not found"
d.write_text(s[:i] + E13 + s[i:])
s = d.read_text()
print("inserted E13 before ### Submission p2_v1")
print("order check:", [h for h in ["### E12:", "### E9-P2:", "### E13:", "### Submission p2_v1"] if h in s],
      "positions:", [s.find(h) for h in ["### E12:", "### E9-P2:", "### E13:", "### Submission p2_v1"]])

inserted E13 before ### Submission p2_v1
order check: ['### E12:', '### E9-P2:', '### E13:', '### Submission p2_v1'] positions: [75407, 71520, 72799, 74295]


In [18]:
from pathlib import Path
d = Path.home() / "IAA" / "DECISIONS.md"
LINE = """- **Public score (Oct 24, 00:08, after the host fixed the scorer): 0.11166**, above the written range 0.06–0.10
  (miss, upward by about 0.012). Most likely cause: the validation fold trains on one year (2023, with the fire
  cluster and an unstable new-business effect); the submission trains on two. Expected ranges for later
  submissions are set relative to this score. First Part 2 submission on the leaderboard; selected by hand.
"""
s = d.read_text()
assert "0.11166" not in s, "score line already there"
k = s.find("- **Kaggle (Oct 23, 15:50): rejected:**")
if k >= 0:
    end = s.find("\n- ", k + 1)                  # next bullet after the 'rejected' item
    nxt = s.find("\n#", k + 1)                   # or the next heading
    cut = min(x for x in (end, nxt) if x >= 0) + 1 if max(end, nxt) >= 0 else len(s)
else:
    i = s.find("### Submission p2_v1")
    assert i >= 0, "p2_v1 section not found"
    nxt = s.find("\n#", i + 1)
    cut = nxt + 1 if nxt >= 0 else len(s)
d.write_text(s[:cut] + LINE + s[cut:])
print("inserted at", cut, "| context:\n", d.read_text()[cut - 200:cut + 120])

inserted at 74815 | context:
 
  relativity; history for 24% of test rows; the fire assumption; a smaller test (more public-score noise
  than Part 1). The range is wide on purpose; later submissions are judged against this one.

- **Public score (Oct 24, 00:08, after the host fixed the scorer): 0.11166**, above the written range 0.06–0.10
  (miss,
